# 04 — Comparer les prévisions en origine glissante

Une règle courte ou un Ridge sur les résidus améliorent-ils les prévisions hebdomadaires par PDQ face à la moyenne sur 12 semaines et au boosting avec historique ?

Les quatre candidats, par ordre de simplicité retenu pour le départage, sont la moyenne des 12 semaines consolidées, le mélange 4/12, Ridge sur les résidus de la moyenne 12 et le boosting historique du notebook 03.

## Protocole fixé avant de comparer

- Au 1er janvier de chaque année de 2019 à 2026, entraîner les modèles sur les semaines dont les résultats auraient déjà été consolidés à cette date selon la règle de disponibilité du projet. Garder les poids fixes pendant l’année et actualiser les entrées chaque semaine. Comparer les quatre candidats sur les mêmes lignes, avec le même horizon.
- Choisir sur 2019–2023. Calculer la MAE sur toutes les observations PDQ-semaine de cette période et retenir le premier candidat dans l’ordre ci-dessus dont la MAE est à au plus `tolerance_mae` de la meilleure. Afficher aussi la MAE par année.
- Utiliser 2024–2026 comme diagnostic déjà consulté. Rapporter le p90 des erreurs absolues, le biais du PDQ 21 (réel moins prévision) en 2024 puis en 2025–2026, et les hausses détectées et fausses alertes. Une hausse correspond à un dépassement de la moyenne 12 d’au moins `seuil_hausse` actes, pour le réel comme pour la prévision.
- Estimer l’incertitude du gain de MAE sur la moyenne 12 par bootstrap apparié en blocs de semaines consécutives, en gardant ensemble les PDQ d’une même semaine. Une fois le choix figé, réentraîner le candidat retenu sur l’historique consolidé disponible et sauvegarder son artefact et son `meta.json`.
- Le rejeu est rétrospectif. Le panel reprend la cohorte du notebook 00 et un export unique. L’horizon limite l’usage des semaines récentes. Il ne vérifie pas la disponibilité réelle des chiffres à chaque date.

In [ ]:
import hashlib
import json
import tomllib
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "configs/crimes.toml").is_file():
    ROOT = ROOT.parent

with (ROOT / "configs/crimes.toml").open("rb") as f:
    cfg = tomllib.load(f)

source = ROOT / cfg["data"]["path"]
artifacts = ROOT / "artifacts"
panel_dir = artifacts / "00_start_here"
meta_panel = json.loads((panel_dir / "meta.json").read_text(encoding="utf-8"))
meta_03 = json.loads(
    (artifacts / "03_historique" / "meta.json").read_text(encoding="utf-8")
)

with source.open("rb") as f:
    empreinte = hashlib.file_digest(f, "sha256").hexdigest()[:16]
if empreinte != meta_panel["source_sha16"] or empreinte != meta_03["source_sha16"]:
    raise ValueError("Le CSV et les artefacts ne correspondent pas. Relance les notebooks 00 à 03.")

for section in ("data", "panel", "split"):
    if cfg[section] != meta_panel["config"][section]:
        raise ValueError(f"La config [{section}] a changé. Relance les notebooks 00 à 03.")

comparaison = cfg["comparaison"]
historique = cfg["historique"]
horizon = historique["horizon"]
if (
    horizon != meta_03["lag_min_semaines"]
    or historique["lags"] != meta_03["lags"]
    or historique["fenetres"] != meta_03["fenetres"]
    or cfg["panel"]["lag_days"] != meta_03["lag_days"]
):
    raise ValueError("Les réglages historiques ont changé. Relance le notebook 03.")

panel = pd.read_parquet(panel_dir / "panel.parquet", columns=["SEMAINE", "PDQ", "y"])
print(f"Panel : {panel.shape} | horizon : {horizon} semaines")
print(f"Référence (notebook 03) : {meta_03['modele_retenu']} | MAE test {meta_03['mae_test']:.3f}")

## Variables disponibles avant la semaine cible

On reprend les lags, les moyennes consolidées et le mois du notebook 03. Les moyennes mobiles commencent à `s−horizon`. Pour Ridge, on calcule deux écarts à la normale de 12 semaines. La cible d’entraînement de Ridge est le résidu `y − moy_12`. Elle sera calculée dans la boucle sur les lignes d’entraînement.

In [ ]:
panel = panel.sort_values(["PDQ", "SEMAINE"]).reset_index(drop=True)
if panel.duplicated(["PDQ", "SEMAINE"]).any():
    raise ValueError("Le panel contient des doublons PDQ-semaine.")

ecart_semaines = panel.groupby("PDQ")["SEMAINE"].diff().dropna()
if not ecart_semaines.eq(pd.Timedelta(weeks=1)).all():
    raise ValueError("Il manque des semaines dans le panel. Vérifie les lags avant de continuer.")

lags = historique["lags"]
fenetres = historique["fenetres"]
if min(lags) < horizon or horizon not in lags or not {4, 12} <= set(fenetres):
    raise ValueError("La comparaison exige le lag à l’horizon et les moyennes 4 et 12.")

par_pdq = panel.groupby("PDQ")["y"]
for lag in lags:
    panel[f"lag_{lag}"] = par_pdq.shift(lag)

for fenetre in fenetres:
    panel[f"moy_{fenetre}"] = par_pdq.transform(
        lambda serie: serie.shift(horizon).rolling(fenetre).mean()
    )

colonnes_historiques = [f"lag_{lag}" for lag in lags] + [
    f"moy_{fenetre}" for fenetre in fenetres
]
panel["mois"] = panel["SEMAINE"].dt.month
panel["ecart_moy_4"] = panel["moy_4"] - panel["moy_12"]
panel["ecart_lag_recent"] = panel[f"lag_{horizon}"] - panel["moy_12"]
panel = panel.dropna(subset=colonnes_historiques).copy()
panel["annee"] = panel["SEMAINE"].dt.year

debut = panel["SEMAINE"].min().date()
fin = panel["SEMAINE"].max().date()
print(f"Lignes : {len(panel)} | {debut} → {fin}")
print(panel.groupby("annee").size().to_string())